# F3 · Mean versus median: OLS and quantile regression

**Type**: Compare · **Level**: ●●● challenge · **Time**: about 45 minutes · **extended path**

## Objective

Compare `LinearRegression` (which estimates the conditional mean) and `QuantileRegressor(quantile=0.5)` (which estimates the
conditional median) on skewed and heavy-tailed noise, and see that each one wins on the metric it optimizes.

## Dataset

Synthetic, from `generate(n, seed, noise)`: `y = 5 + 2·x + ε` with x uniform in [0, 10] and `ε` lognormal (skewed, mean
1.65, median 1) or Student `t` with 2 degrees of freedom (very heavy tails, infinite variance). The function
also returns `eps`, so you can look at the true noise.

## Educational purpose

Connect lesson 2.12 (MAE versus RMSE, mean versus median) to an experiment. The model is not a neutral
container: **the loss function decides what it estimates**. You will also see a situation in which the median is
much more stable than the mean.

## Task

1. With `X_tr, y_tr, eps_tr = generate(500, 0)` (lognormal noise) train `ols = LinearRegression()` and `median = QuantileRegressor(quantile=0.5, alpha=0, solver="highs")`. Save `ols_intercept`, `med_intercept`, `ols_slope`, `med_slope`.
2. Compute `theoretical_gap = eps_tr.mean() − np.median(eps_tr)` and `intercept_gap = ols_intercept − med_intercept`. They should be similar: OLS learns "5 + mean of the noise", median regression "5 + median of the noise".
3. With `X_te, y_te, _ = generate(5000, 1)` compute `mae_ols`, `mae_med`, `rmse_ols`, `rmse_med` on the test set.
4. Simulate 200 times (`seed = 100 + s`, `generate(100, seed, "t")`) the two slopes with heavy-tailed noise and save `std_slope_ols` and `std_slope_med`, the standard deviation of the 200 estimates of each method.
5. Assign to `true_false` a dictionary for the three statements:
   - **a**: `LinearRegression` estimates the conditional mean, quantile regression with quantile 0.5 the conditional median;
   - **b**: with skewed noise the two lines have a very different slope;
   - **c**: a model can have the better MAE and the worse RMSE than another on the same data.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LinearRegression, QuantileRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error


def generate(n, seed, noise="lognormal"):
    rng = np.random.default_rng(seed)
    x = rng.uniform(0, 10, n)
    eps = rng.lognormal(0, 1, n) if noise == "lognormal" else rng.standard_t(2, n)
    return x.reshape(-1, 1), 5 + 2 * x + eps, eps

## Your work

In [ ]:
X_tr, y_tr, eps_tr = generate(500, 0)
ols = ...
median = ...
ols_intercept, med_intercept = ..., ...
ols_slope, med_slope = ..., ...

In [ ]:
theoretical_gap = ...
intercept_gap = ...
print(f"mean - median of the noise: {theoretical_gap:.3f} | difference between the two intercepts: {intercept_gap:.3f}")

In [ ]:
X_te, y_te, _ = generate(5000, 1)
mae_ols, mae_med = ..., ...
rmse_ols, rmse_med = ..., ...

In [ ]:
slopes_ols, slopes_med = [], []
for s in range(200):
    ...
std_slope_ols = ...
std_slope_med = ...

In [ ]:
true_false = {"a": ..., "b": ..., "c": ...}

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`QuantileRegressor(quantile=0.5, alpha=0, solver="highs")` has `intercept_` and `coef_` like the other models. In the loop: `X, y, _ = generate(100, 100 + s, "t")`, train the two models and append `coef_[0]` to the two lists.'):
    check(abs(ols_slope - 2) < 0.15 and abs(med_slope - 2) < 0.15,
          "the two slopes are both close to the true value 2 (the noise is independent of x)",
          "both slopes should be about 2")
    check(abs(intercept_gap - theoretical_gap) < 0.4 and intercept_gap > 0.2,
          "the two intercepts differ by about mean - median of the noise",
          "OLS and median regression must have intercepts that differ by about (mean - median) of the noise")
    check(mae_med < mae_ols, "median regression wins on the MAE", "on the MAE median regression should win")
    check(rmse_ols < rmse_med, "OLS wins on the RMSE", "on the RMSE OLS should win")
    check(len(slopes_ols) == 200 and len(slopes_med) == 200, "200 estimates per method", "you need 200 slopes for each method")
    check(std_slope_ols > 1.3 * std_slope_med, "with heavy tails the OLS estimate of the slope is much less stable",
          "with t(2) noise the standard deviation of OLS should exceed by at least 30% that of median regression")
    check(true_false == {"a": True, "b": False, "c": True}, "statements judged correctly",
          "the two lines differ in the intercept, not in the slope")
done()

---
The commented solution is in `solutions/F_compare/F3_mean_versus_median_ols_and_quantile_regression.ipynb`. Open it after you pass the check, or after you have really tried.